## Bijdragen en GenAI

**Jorre Van Dyck:** pipeline opgesteld, gezamenlijke backend ontwikkeld en UI gebouwd, met substantiële hulp van Codex voor implementatie en controles. De oorspronkelijke Mushroom-modellen van Andrew en Citi Bike-uitwerking/inference van Raoul behouden hun bronvermelding.

Jorres bevestigde projectbijdragen: pipeline, gezamenlijke backend, UI en AWS-training. Zie [bijdragen](../BIJDRAGEN.md) en [GenAI Disclosure](../GENAI_DISCLOSURE.md), inclusief vijf echte prompts en uitgevoerde controles. De overige teamleden bevestigen hun eigen aandeel en zelfstandige beheersing.


Final-leesversie; oorspronkelijke bestanden en hun uitvoer blijven behouden. Jorres bijdrage en de GenAI-policy zijn nu verwerkt in de centrale verantwoording. Bron: `secondary_mushroom/10_deploy.ipynb`.

# 10 — Gekozen Mushroom-model deployen en controleren

## Bijdragen
- Andrew: oorspronkelijke Mushroom-analyse en modellen.
- Jorre: pipeline opgesteld, gezamenlijke backend ontwikkeld, UI gebouwd en SageMaker-model getraind; met substantiële hulp van Codex.
- Codex: verliesvrije export, API/formuliermigratie, controles en deze documentatie.
- Het team reviewt de code, modelkeuze en GenAI-vermelding en verdedigt het werk.

## Doel en gekozen model
Na [de vergelijking](09_vergelijking.ipynb) gebruiken we RF300: 300 bomen,
maximale diepte 20 en gebalanceerde klassegewichten. Dit model is gekozen op
vijfvoudige CV macro F1 (0,761876) op 4.100 trainingsrijen. De 900 ontwikkeltest-
records geven 81,33% accuracy en 64,52% giftige recall. De testset was eerder
bekeken; deze cijfers zijn geen onaangeraakte eindtest.

Bron: [de cursus-CSV met 5.000 rijen](../../MushroomFinal/mushroom_project_dataset.csv).
`jumbled_noise_0` is verwijderd; `jumbled_noise_1` blijft behouden. De pipeline
is niet op de 900 testrecords meegetraind. Het oorspronkelijke 61k-UCI-experiment
is historisch en wordt hier niet gebruikt.

Dit notebook traint niets. Het controleert de artifactketen, lokale inference,
de gepubliceerde backend en de bijbehorende websitegegevens.

## Beslisregel aangepast na controle van hoge modelscores

Jorre wil giftige gevallen niet als eetbaar laten bevestigen. Onder de bestaande
trainingsvalidatievoorspellingen met minstens 95% eetbare modelscore zijn 55
gevallen, waarvan zeven werkelijk giftig. Zelfs bij minstens 99% is één van de
vier gevallen giftig. Een gekozen percentage is dus geen bewezen kans of
veiligheidsgarantie. Zie [de volledige analyse](../../docs/mushroom_decisions.md).

De website toont vanaf 50% giftigheidsscore **Model voorspelt: giftig** en onder
50% **Onzeker / mogelijk giftig**. Er is geen bevestiging van eetbaarheid.
De ruwe e/p-klasse en scores blijven onveranderd voor de vergelijking; onzekere
gevallen worden niet als correcte giftig-voorspellingen meegeteld.
API en frontend bewaken `mushroom_no_edibility_confirmation_v1`.
De controles hieronder zijn opnieuw uitgevoerd met deze beslisregel.

## 1. Controleer welk artifact wordt aangeboden

Het gekozen sklearn-1.4.2-model staat in `comparison_correct`. De bestaande
Python-3.13-backend laadt een verliesvrije NumPy-export met exact dezelfde
preprocessing, splits en bladwaarden, zodat we geen oude sklearn-pickle in een
andere libraryversie hoeven te laden. De bronhash en exportaudit verbinden beide
bestanden. De volgende cel controleert deze herkomst en de elf velden.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'backend/main.py').exists())
sys.path.insert(0, str(ROOT))
from backend.mushroom_model import PortableMushroom, MODEL_ID, FEATURES
from backend.mushroom_policy import apply_policy, POLICY_ID

folder = ROOT/'MushroomFinal/comparison_correct'
summary = json.loads((folder/'comparison_summary.json').read_text(encoding='utf-8'))
audit = json.loads((ROOT/'backend/artifacts/mushroom_export_audit.json').read_text(encoding='utf-8'))
metadata = json.loads((ROOT/'frontend/metadata.json').read_text(encoding='utf-8'))
model = PortableMushroom()
assert hashlib.sha256((folder/'selected_pipeline.joblib').read_bytes()).hexdigest() == summary['artifact']['sha256']
assert model.metadata['source_pipeline_sha256'] == audit['source_pipeline_sha256'] == hashlib.sha256((ROOT/model.metadata['source_pipeline']).read_bytes()).hexdigest()
assert metadata['model_id'] == MODEL_ID == 'mushroom_final_rf300'
assert {f['name'] for f in metadata['fields']} == set(FEATURES)
assert len(FEATURES) == 11 and 'jumbled_noise_0' not in FEATURES
assert audit['label_disagreements'] == 0 and audit['max_probability_difference'] < 1e-12
print('Herkomst en formulier gecontroleerd:', MODEL_ID)
print('Bronpipeline SHA-256:', audit['source_pipeline_sha256'])
print('Exportaudit:', audit)

Herkomst en formulier gecontroleerd: mushroom_final_rf300
Bronpipeline SHA-256: 6410b277e49c3706fa4d70db548e2545066a78bcaa6406aa83fef418c489c626
Exportaudit: {'source_pipeline_sha256': '6410b277e49c3706fa4d70db548e2545066a78bcaa6406aa83fef418c489c626', 'model_id': 'mushroom_final_rf300', 'test_rows_verified': 900, 'additional_rows_verified': 5000, 'label_disagreements': 0, 'max_probability_difference': 2.220446049250313e-16, 'tolerance': 1e-12, 'export_sklearn': '1.4.2', 'forest_sha256': 'bb562ebed41e524ae0f61dae9dc4f0ce5a321530752534fd0c4d357650dc5f65'}


## 2. Controleer inference in de servingomgeving

De exporter heeft het oorspronkelijke model met de export vergeleken op alle
5.000 bronrijen en zes extra gevallen, waaronder nulmetingen, missende waarden en
een onbekende categorie. Hier controleren we onafhankelijk alle 900 opgeslagen
testlabels en kansen met de huidige servinglibraries. Er wordt niet getuned.
De ruwe modelklasse is 0=e en 1=p. De zichtbare boodschap volgt apart de voorzichtige beslisregel.

In [2]:
data = pd.read_csv(ROOT/'MushroomFinal/mushroom_project_dataset.csv')
data.columns = data.columns.str.replace('-', '_', regex=False)
saved = pd.read_csv(folder/'test_predictions.csv', float_precision='round_trip')
X_test = data[FEATURES].iloc[saved.source_row]
proba = model.predict_proba(X_test)
np.testing.assert_array_equal(np.argmax(proba, axis=1), saved.rf_final_prediction)
np.testing.assert_allclose(proba[:, 1], saved.rf_final_probability_p, rtol=0, atol=1e-12)
assert len(X_test) == 900
print('Alle 900 labels en kansen komen overeen met de oorspronkelijke pipeline.')
print('Voorzichtige boodschap:', apply_policy(model.predict_record(metadata['example'])))

Alle 900 labels en kansen komen overeen met de oorspronkelijke pipeline.
Voorzichtige boodschap: {'label': 'e', 'prediction': 'Onzeker / mogelijk giftig', 'probability_poisonous': 0.03465518938833522, 'decision': 'uncertain', 'decision_policy_id': 'mushroom_no_edibility_confirmation_v1'}


## 3. API en formulier

`backend/main.py` laadt het model eenmaal bij startup. `POST /predict/mushroom`
valideert elf underscore-velden: tien verplichte nullable kenmerken en het
optionele `jumbled_noise_1`. Extra velden, onbekende codes en negatieve metingen
geven 422. Bij een ontbrekend model geeft inference 503. Nul blijft een meting;
ontbrekende waarden worden door de oorspronkelijke preprocessing ingevuld.

De response bevat model-ID, ruwe e/p-klasse, beslisregel-ID, een voorzichtige boodschap en modelscore voor
de giftige klasse en een educatieve disclaimer. De boodschap bevestigt geen eetbaarheid, ook niet bij hoge scores. De website bouwt het formulier
uit dezelfde metadata en controleert de model-ID bij health en voorspelling.
De projectpagina toont de resultaten van deze specifieke modelversie.

Lokaal starten vanaf de repository-root (aparte terminal):

```powershell
python -m pip install -r backend/requirements-dev.txt
python -m uvicorn backend.main:app --reload
```

Open `http://127.0.0.1:8000/` voor het formulier of `/docs` voor Swagger.
Dezelfde API blijft Raouls gekozen Citi Bike-model aanbieden.

## 4. Hosting, herstart en updates via GitHub

- Backend en ingebouwde frontend: https://deeplearningteam10-api.onrender.com/
- Afzonderlijke teamfrontend: https://10messi10.netlify.app/
- Repository: https://github.com/JorreVanDyck10/DeepLearningTeam10

De gekoppelde main-branch activeert Render en Netlify bij een push. Render
installeert `backend/requirements.txt` en start
`uvicorn backend.main:app --host 0.0.0.0 --port $PORT`; healthcheck is `/health`.
Netlify publiceert `frontend/`. Deze frontend kiest automatisch de Render-URL;
de backend staat haar origin expliciet toe. Een open pagina moet na deploy
worden vernieuwd. Een koude Render-start kan extra tijd kosten.

`MUSHROOM_RF300_MANIFEST` kan het standaardmanifestpad aanpassen. Oude
`ANDREW_MUSHROOM_MODEL_PATH`-instellingen selecteren RF200 niet meer. De bestaande
Python 3.13-runtime en dependencies blijven bruikbaar. Gebruik in het Render-
dashboard **Manual Deploy → Deploy latest commit** voor opnieuw publiceren van
dezelfde versie; lokaal herstart je uvicorn.

GitHub Actions controleert API-routes, de 900 referentievoorspellingen, Citi Bike,
JavaScript en reproduceerbare frontendmetadata. Render staat op On Commit:
het starten van een deployment wacht dus niet automatisch op deze CI-uitkomst.

Bij een bewust gekozen nieuw getraind model: voer eerst de vergelijking uit,
pas de gepinde bronhash/schema indien nodig aan, maak de serving-export in de
Python 3.11-vergelijkingsomgeving en exporteer daarna frontendmetadata. Commit
model, manifest, audit en metadata samen. Een push vervangt het geladen model
via herstart van de deployment; hij traint geen nieuw model op nieuwe data.
Dit is automatische modelpublicatie, geen automatische hertrainingspipeline.
Zie [de exportinstructies](../../docs/mushroom_rf300.md) en
[algemene deploymentinstellingen](../../docs/deployment.md).

## 5. Controleer de echte publieke routes

Deze cel haalt de actuele Netlify-metadata en Render-health op, voert een echte
voorspelling uit en vergelijkt die met lokale inference. Ook de bestaande Citi
Bike-demonstratie moet 24 uurvoorspellingen opleveren. De uitvoer is bewijs van
het controlemoment, geen garantie dat een externe gratis service altijd online
blijft. Bij een versieverschil wacht je op beide deployments en voer je opnieuw uit.

In [3]:
import urllib.request
from datetime import datetime, timezone

API = 'https://deeplearningteam10-api.onrender.com'
SITE = 'https://10messi10.netlify.app'
def request_json(url, payload=None):
    body = json.dumps(payload).encode() if payload is not None else None
    request = urllib.request.Request(url, data=body, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(request, timeout=120) as response:
        return json.load(response)

health = request_json(API+'/health')
live_metadata = request_json(SITE+'/metadata.json')
assert health['mushroom_model_loaded'] and health['citibike_model_loaded']
assert health['mushroom_model_id'] == live_metadata['model_id'] == MODEL_ID
for key in ['model_id','source_pipeline_sha256','example','fields','dataset_rows']:
    assert live_metadata[key] == metadata[key], key
for key in ['accuracy','recall_poisonous','poisonous_predicted_edible']:
    assert abs(live_metadata['metrics'][key]-metadata['metrics'][key]) < 1e-12, key
actual = request_json(API+'/predict/mushroom', metadata['example'])
expected = model.predict_record(metadata['example'])
assert actual['model_id'] == MODEL_ID and actual['label'] == expected['label']
assert actual['decision_policy_id'] == POLICY_ID
assert actual['decision'] == apply_policy(expected)['decision']
assert actual['prediction'] != 'eetbaar'
assert abs(actual['probability_poisonous']-expected['probability_poisonous']) < 1e-12
city = request_json(API+'/predict/citibike', {'date':'2026-04-30', 'hour':8, 'mode':'demo'})
assert city['model_id'] == 'raoul_tuned_random_forest_320' and len(city['daily_predictions']) == 24
print('Live controle UTC:', datetime.now(timezone.utc).isoformat())
print('Health:', health)
print('Mushroom:', actual)
print('Citi Bike:', city['model_id'], len(city['daily_predictions']), 'uurvoorspellingen')

Live controle UTC: 2026-10-14T17:18:59.367350+00:00
Health: {'status': 'ok', 'mushroom_model_loaded': True, 'citibike_model_loaded': True, 'mushroom_model_id': 'mushroom_final_rf300', 'mushroom_model_revision': '6410b277e49c3706fa4d70db548e2545066a78bcaa6406aa83fef418c489c626', 'citibike_model_id': 'raoul_tuned_random_forest_320'}
Mushroom: {'model_id': 'mushroom_final_rf300', 'model_revision': '6410b277e49c3706fa4d70db548e2545066a78bcaa6406aa83fef418c489c626', 'label': 'e', 'prediction': 'Onzeker / mogelijk giftig', 'probability_poisonous': 0.03465518938833522, 'disclaimer': 'Educatieve voorspelling op hypothetische data; niet gebruiken om echte paddenstoelen te eten.', 'decision': 'uncertain', 'decision_policy_id': 'mushroom_no_edibility_confirmation_v1'}
Citi Bike: raoul_tuned_random_forest_320 24 uurvoorspellingen


## 6. Eindcontrole en conclusie

Test op de website **Voorbeeld invullen → Voorspellen** en bekijk de metrics
onder Project. Test daarna City Bike met de historische demonstratie. Controleer
bij een wijziging zowel de netwerkrespons als de zichtbare melding. De volledige
regressiecontroles staan onder `tests/` en draaien in GitHub Actions.

RF300 is gekoppeld aan backend en frontend met de oorspronkelijke preprocessing
en gevalideerde labels/kansen. De website presenteert de 900 ontwikkeltest-
resultaten en bewaakt de modelversie. Het model mist nog giftige voorbeelden;
de demonstratie is uitsluitend educatief. Teamreview, eventuele resterende
opdrachtvereisten en de definitieve inlevering blijven de verantwoordelijkheid
van het team.